# 第 23 章互动实验：晶圆图与空间模式

画出 300 mm 晶圆的 die 网格，合成几种典型的缺陷空间分布，用径向密度和边缘比例区分它们。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "23_半导体检测领域知识"
ASSIGNMENT = "23-1_晶圆图.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
grid = attempt(mod.die_grid, 300.0, 12.0, 14.0, 3.0, (-6.0, -7.0))
if grid is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    for col, row, cx, cy in grid:
        ax.add_patch(plt.Rectangle((cx - 6, cy - 7), 12, 14, fill=False, lw=0.5))
    ax.add_patch(plt.Circle((0, 0), 150, fill=False, color="k")); ax.add_patch(plt.Circle((0, 0), 147, fill=False, ls="--", color="gray"))
    ax.set_xlim(-155, 155); ax.set_ylim(-155, 155); ax.set_aspect("equal"); ax.set_title(f"完整 die 数：{len(grid)}"); plt.show()

In [ ]:
rng = np.random.default_rng(0)
def sample_disk(n, rmax=147):
    r = rmax * np.sqrt(rng.random(n)); t = rng.random(n) * 2 * np.pi
    return np.stack([r * np.cos(t), r * np.sin(t)], 1)
patterns = {
    "随机": sample_disk(400),
    "中心": rng.normal(scale=35, size=(400, 2)),
    "边缘环": (lambda r, t: np.stack([r * np.cos(t), r * np.sin(t)], 1))(rng.uniform(132, 147, 400), rng.random(400) * 2 * np.pi),
    "划痕": np.stack([np.linspace(-90, 60, 400) + rng.normal(scale=1.5, size=400), 0.6 * np.linspace(-90, 60, 400) + 20 + rng.normal(scale=1.5, size=400)], 1),
}
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for i, (name, p) in enumerate(patterns.items()):
    ax[0, i].scatter(p[:, 0], p[:, 1], s=2); ax[0, i].add_patch(plt.Circle((0, 0), 150, fill=False))
    ax[0, i].set_xlim(-155, 155); ax[0, i].set_ylim(-155, 155); ax[0, i].set_aspect("equal"); ax[0, i].set_title(name)
    dens = attempt(mod.radial_density, p, 150.0, 6)
    er = attempt(mod.edge_ratio, p, 150.0, 15.0)
    if dens is not None:
        ax[1, i].bar(np.arange(6), dens); ax[1, i].set_title(f"径向密度｜边缘比例 {er:.2f}")
plt.show()

## die 级缺陷计数热图

把缺陷分配到 die，画出每个 die 的缺陷数，这就是良率工程师每天看的图。

In [ ]:
if grid is not None:
    pts = np.concatenate([patterns["随机"][:150], patterns["边缘环"][:200]])
    ids = attempt(mod.assign_to_die, pts, 12.0, 14.0, (-6.0, -7.0))
    if ids is not None:
        counts = {}
        for c, r in map(tuple, ids): counts[(c, r)] = counts.get((c, r), 0) + 1
        fig, ax = plt.subplots(figsize=(6, 6))
        for col, row, cx, cy in grid:
            n = counts.get((int(col), int(row)), 0)
            ax.add_patch(plt.Rectangle((cx - 6, cy - 7), 12, 14, color=plt.cm.Reds(min(n, 5) / 5), ec="k", lw=0.3))
        ax.set_xlim(-155, 155); ax.set_ylim(-155, 155); ax.set_aspect("equal"); ax.set_title("每个 die 的缺陷数"); plt.show()
        bad = sum(1 for col, row, _, _ in grid if counts.get((int(col), int(row)), 0) > 0)
        print(f"有缺陷的完整 die：{bad}/{len(grid)}，若每个缺陷都致命，良率上限约 {1 - bad / len(grid):.1%}")

## 记录

- 哪两种模式只靠径向密度分不开？还需要什么特征？
- 把入职前术语清单整理进 `实验记录.md`。